Loading saved model

In [1]:
import pickle

with open('models/lin_reg.bin', 'rb') as f:
    dv, lr = pickle.load(f)

print("Model loaded successfully!")
print(type(dv), type(lr))

Model loaded successfully!
<class 'sklearn.feature_extraction._dict_vectorizer.DictVectorizer'> <class 'sklearn.linear_model._base.LinearRegression'>


In [2]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 35.1 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 41.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52/52 [mlflow]52 [mlflow]52 [mlflow-skinny]tracing]dk]ons]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-trip-duration")

2026/05/17 03:05:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/17 03:05:22 INFO mlflow.store.db.utils: Updating database tables
2026/05/17 03:05:23 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-trip-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/nyc-taxi-trip-analysis/mlruns/1', creation_time=1778987123795, experiment_id='1', last_update_time=1778987123795, lifecycle_stage='active', name='nyc-taxi-trip-duration', tags={}, trace_location=None, workspace='default'>

Re-running training but now tracked with MLflow

In [7]:
import pandas as pd

In [8]:
df_jan = pd.read_parquet('data/green_tripdata_2026-01.parquet')
df_feb = pd.read_parquet('data/green_tripdata_2026-02.parquet')

def prepare_data(df):
    df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)].copy()
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

df_jan = prepare_data(df_jan)
df_feb = prepare_data(df_feb)

In [9]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

train_dicts = df_jan[categorical + numerical].to_dict(orient='records')
val_dicts = df_feb[categorical + numerical].to_dict(orient='records')

X_train = dv.transform(train_dicts)   # transform only, dv already fitted
X_val = dv.transform(val_dicts)

y_train = df_jan['duration'].values
y_val = df_feb['duration'].values

In [10]:
import pickle
import numpy as np
from sklearn.metrics import mean_squared_error
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-trip-duration")

# Load pickled model
with open('models/lin_reg.bin', 'rb') as f:
    dv, lr = pickle.load(f)

with mlflow.start_run():

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", "PULocationID, DOLocationID, trip_distance")
    mlflow.log_param("train_month", "2026-01")
    mlflow.log_param("val_month", "2026-02")

    # Evaluate — no training needed, model already trained!
    y_pred_train = lr.predict(X_train)
    y_pred_val = lr.predict(X_val)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

    # Log metrics
    mlflow.log_metric("rmse_train", rmse_train)
    mlflow.log_metric("rmse_val", rmse_val)

    # Log the already trained model
    mlflow.sklearn.log_model(lr, "model")

    print(f"RMSE Train: {rmse_train:.2f}")
    print(f"RMSE Val:   {rmse_val:.2f}")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

2026/05/17 03:10:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 03:10:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE Train: 8.05
RMSE Val:   8.34
Run ID: 05fbc0e0283848869ec9db084ce004de


Launching UI

In [11]:
import subprocess
subprocess.Popen(["mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db", "--port", "5000"])
print("MLflow UI starting at port 5000...")

MLflow UI starting at port 5000...
Registry store URI not provided. Using backend store URI.


[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/17 03:11:27 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/17 03:11:27 INFO:     Started parent process [24596]
2026/05/17 03:11:49 INFO:     Started server process [24678]
2026/05/17 03:11:49 INFO:     Waiting for application startup.
2026/05/17 03:11:49 INFO:     Application startup complete.
2026/05/17 03:11:50 INFO:     Started server process [24679]
2026/05/17 03:11:50 INFO:     Waiting for application startup.
2026/05/17 03:11:50 INFO:     Application startup complete.
2026/05/17 03:11:50 INFO:     Started server process [24680]
2026/05/17 03:11:50 INFO:     Waiting for application startup.
2026/05/17 03:11:50 INFO:     Application startup complete.
2026/05/17 03:11:51 INFO:     Started server process [24677]
2026/05/17 03:11:51 INFO:     Waiting f

2026/05/17 03:12:13 INFO:     127.0.0.1:59116 - "GET / HTTP/1.1" 200 OK
2026/05/17 03:12:13 INFO:     127.0.0.1:59116 - "GET /static-files/static/js/main.22ec8c1e.js HTTP/1.1" 200 OK
2026/05/17 03:12:13 INFO:     127.0.0.1:59132 - "GET /static-files/static/css/main.b0f58ef9.css HTTP/1.1" 200 OK
2026/05/17 03:12:14 INFO:     127.0.0.1:59136 - "GET /static-files/manifest.json HTTP/1.1" 200 OK
2026/05/17 03:12:15 INFO:     127.0.0.1:59116 - "GET /static-files/TelemetryLogger.telemetry-worker.d0f8ea2c83c08b63528f.worker.js HTTP/1.1" 200 OK
2026/05/17 03:12:15 INFO:     127.0.0.1:59132 - "GET /static-files/favicon.ico HTTP/1.1" 200 OK
2026/05/17 03:12:15 INFO:     127.0.0.1:59116 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
2026/05/17 03:12:16 INFO:     127.0.0.1:59136 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
2026/05/17 03:12:16 INFO:     127.0.0.1:59136 - "GET /static-files/static/js/4205.9e750e7b.chunk.js HTTP/1.1" 200 OK
2026/05/17 03:12:16 INFO:     127.0.0.

In [12]:
from sklearn.linear_model import Lasso

with mlflow.start_run():
    mlflow.log_param("model_type", "Lasso")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_param("train_month", "2026-01")
    mlflow.log_param("val_month", "2026-02")

    lasso = Lasso(alpha=0.1)
    lasso.fit(X_train, y_train)

    y_pred_train = lasso.predict(X_train)
    y_pred_val = lasso.predict(X_val)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

    mlflow.log_metric("rmse_train", rmse_train)
    mlflow.log_metric("rmse_val", rmse_val)
    mlflow.sklearn.log_model(lasso, "model")

    print(f"Lasso RMSE Train: {rmse_train:.2f}")
    print(f"Lasso RMSE Val:   {rmse_val:.2f}")

2026/05/17 03:22:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 03:22:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso RMSE Train: 9.80
Lasso RMSE Val:   9.86


In [13]:
from sklearn.linear_model import Ridge

with mlflow.start_run():
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", 0.1)
    mlflow.log_param("train_month", "2026-01")
    mlflow.log_param("val_month", "2026-02")

    ridge = Ridge(alpha=0.1)
    ridge.fit(X_train, y_train)

    y_pred_train = ridge.predict(X_train)
    y_pred_val = ridge.predict(X_val)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

    mlflow.log_metric("rmse_train", rmse_train)
    mlflow.log_metric("rmse_val", rmse_val)
    mlflow.sklearn.log_model(ridge, "model")

    print(f"Ridge RMSE Train: {rmse_train:.2f}")
    print(f"Ridge RMSE Val:   {rmse_val:.2f}")

2026/05/17 03:22:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 03:22:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge RMSE Train: 8.02
Ridge RMSE Val:   8.34
2026/05/17 03:22:38 INFO:     127.0.0.1:52662 - "GET / HTTP/1.1" 304 Not Modified
2026/05/17 03:22:40 INFO:     127.0.0.1:52662 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
2026/05/17 03:22:40 INFO:     127.0.0.1:52666 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
2026/05/17 03:22:40 INFO:     127.0.0.1:52666 - "POST /graphql HTTP/1.1" 200 OK
2026/05/17 03:22:40 INFO:     127.0.0.1:52662 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/17 03:22:40 INFO:     127.0.0.1:52672 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
2026/05/17 03:22:41 INFO:     127.0.0.1:52662 - "GET /ajax-api/2.0/mlflow/model-versions/search?filter=run_id%3D%2705fbc0e0283848869ec9db084ce004de%27 HTTP/1.1" 200 OK
2026/05/17 03:22:41 INFO:     127.0.0.1:52672 - "POST /graphql HTTP/1.1" 200 OK
2026/05/17 03:22:41 INFO:     127.0.0.1:52672 - "POST /ajax-api/2.0/mlflow/logged-models/search HTTP/1.1" 200 OK
2026/05/17 0

Registering best model (Ridge)

In [14]:
from mlflow.tracking import MlflowClient

client = MlflowClient("sqlite:///mlflow.db")

# Find all runs and get the best one by rmse_val
runs = client.search_runs(
    experiment_ids="1",
    order_by=["metrics.rmse_val ASC"]
)

best_run = runs[0]
print(f"Best Run ID: {best_run.info.run_id}")
print(f"Best RMSE Val: {best_run.data.metrics['rmse_val']:.2f}")
print(f"Model type: {best_run.data.params['model_type']}")

Best Run ID: 05fbc0e0283848869ec9db084ce004de
Best RMSE Val: 8.34
Model type: LinearRegression


In [15]:
best_run_id = best_run.info.run_id

model_uri = f"runs:/{best_run_id}/model"
model_name = "nyc-taxi-trip-duration"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"Model registered: {registered_model.name}")
print(f"Version: {registered_model.version}")

Successfully registered model 'nyc-taxi-trip-duration'.
2026/05/17 03:28:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 05fbc0e0283848869ec9db084ce004de has no artifacts at artifact path 'model', registering model based on models:/m-94eb1556946246e5ad18cbb2ba955550 instead


Model registered: nyc-taxi-trip-duration
Version: 1


Created version '1' of model 'nyc-taxi-trip-duration'.


Promoting to production

In [16]:
client.set_registered_model_alias(
    name=model_name,
    alias="production",
    version=registered_model.version
)

print(f"Model version {registered_model.version} promoted to production! 🚀")

Model version 1 promoted to production! 🚀


Loading and using the production model

In [17]:
production_model = mlflow.sklearn.load_model(f"models:/{model_name}@production")

y_pred = production_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Production model RMSE: {rmse:.2f}")

Production model RMSE: 8.34
2026/05/17 03:30:54 INFO:     127.0.0.1:37608 - "GET / HTTP/1.1" 304 Not Modified
2026/05/17 03:30:55 INFO:     127.0.0.1:37618 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
2026/05/17 03:30:55 INFO:     127.0.0.1:37608 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
2026/05/17 03:30:55 INFO:     127.0.0.1:37608 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
2026/05/17 03:30:56 INFO:     127.0.0.1:37608 - "GET /ajax-api/2.0/mlflow/experiments/get?experiment_id=1 HTTP/1.1" 200 OK
2026/05/17 03:30:56 INFO:     127.0.0.1:37618 - "GET /ajax-api/2.0/mlflow/runs/get?run_id=51293b666cb948d8ba17bca478a291f3 HTTP/1.1" 200 OK
2026/05/17 03:30:56 INFO:     127.0.0.1:37630 - "GET /ajax-api/2.0/mlflow/runs/get?run_id=eaefd2d288f24b2aafd38fb52a9c2cb8 HTTP/1.1" 200 OK
2026/05/17 03:30:56 INFO:     127.0.0.1:37644 - "GET /ajax-api/2.0/mlflow/runs/get?run_id=05fbc0e0283848869ec9db084ce004de HTTP/1.1" 200 OK
2026/05/17 03:30:56 INFO:     12

In [18]:
production_model = mlflow.sklearn.load_model(f"models:/{model_name}@production")

y_pred = production_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Production model RMSE: {rmse:.2f}")

Production model RMSE: 8.34
2026/05/17 03:35:14 INFO:     127.0.0.1:45474 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/17 03:35:45 INFO:     127.0.0.1:47840 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/17 03:36:16 INFO:     127.0.0.1:41770 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/17 03:36:47 INFO:     127.0.0.1:33874 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
